# Iteris — Phase-B ATTENTION-net Ceiling Diagnostic (CAMUS)

Measures, per CAMUS class, **how much reachable headroom contour refinement has over the
Phase-B (low-data) ATTENTION U-Net** — the one number that decides whether the DRL agent can
possibly beat this baseline, independent of any policy/training issue.

Key outputs (per class, from `headroom_report`):
- **baseline_init_dice** — the Phase-B attention net's own Dice (what we must beat).
- **contour_repr_ceiling** — best Dice a 32-point spline can even represent for this GT.
- **oracle_greedy_ceiling** — best Dice reachable by the contour ACTION SET with GT-optimal
  moves (GT-privileged, NOT deployable). This is the true ceiling.
- **HEADROOM = oracle_greedy_ceiling − baseline_init_dice** — the decisive number:
  > 0.02 → a real win is on the table; ~0 → contour refinement cannot beat this baseline.
- **error decomposition** (boundary vs topology/interior) and **prob_map** calibration.

CPU-only. Warm-start runs the attention net once per class (~5 min).

**Attach (Datasets panel):** `iteris-pkg` (latest), `CAMUS`, and the **Phase-B ATTENTION**
checkpoint dataset (`camus_lf10_best.pt`; the auto-detect falls back to any non-lite
`camus*_best.pt` and warns).


## 0 · Install + locate package

In [ ]:
import subprocess, sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached. Add it in the right-hand panel -> Datasets.')
PKG_ROOT = init_files[0].parent.parent
subprocess.run(['pip', 'install', '-r', str(PKG_ROOT / 'requirements.txt'),
                '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))
print(f'Package at: {PKG_ROOT}')

## 1 · Imports, config, and report helpers

`ATTN_DICE` is the attention U-Net's test Dice per class (the fixed competitor) — used to derive the
*realistic*, non-GT-privileged headroom. `full_report(label)` runs every section below for one class
from its cached warm-start samples and returns a row of summary metrics.

In [ ]:
import numpy as np
import pandas as pd
import scipy.ndimage as ndi

from iteris.config import load_drl_class_config, resolve_agent_config, load_config
from iteris.utils import model_suffix
from iteris.warm_start import precompute_init_masks
from iteris.geometry import dice_score, STRUCT
from iteris.diagnostics import (prob_map_informativeness, error_type_audit,
                                headroom_report, repr_ceiling, oracle_greedy,
                                _base_env_kwargs)

# Attention U-Net competitor test Dice (see CONTEXT.md / SKILLS.md).
# We are diagnosing the ATTENTION net itself, so there is no higher DL baseline
# to compare against -> None everywhere makes headroom_report print the
# GT-oracle (reachable-ceiling) headroom verdict, which is what we want here.
ATTN_DICE = {
    'CAMUS_LV_endo': None, 'CAMUS_LV_epi': None, 'CAMUS_LA': None,
    'BRISC_tumor': None,
    'BRISC_glioma': None, 'BRISC_meningioma': None, 'BRISC_pituitary': None,
}

SAMPLES = {}   # label -> dict(train, val, test)
CFGS    = {}   # label -> resolved cfg (geometry source = TD3 block)
RESULTS = {}   # label -> summary-metrics row


# Phase-B low-data fraction — matches notebooks/phaseB/unet/03_camus_attnunet
# (CAMUS 0.10). Change here to diagnose a different Phase-B subset.
LABEL_FRAC = 0.10

def load_class(config_path, root_names, label):
    """Warm-start the PHASE-B ATTENTION U-Net for one class; cache samples.

    Unlike the lite-net diagnostic (00_...), this points the baseline at the
    ATTENTION config (camus.yaml / brisc.yaml) and sets label_frac=LABEL_FRAC so
    model_suffix resolves the Phase-B low-data attention checkpoint (e.g.
    camus_lf10_best.pt). Everything downstream (headroom/oracle/error decomp) is
    unchanged — it just characterises a different init mask."""
    cfg_full = load_drl_class_config(str(PKG_ROOT / 'configs' / config_path))
    cfg = resolve_agent_config(cfg_full, 'TD3')   # TD3 block carries full geometry
    # ATTENTION baseline (NOT the lite net), at the Phase-B low-data regime.
    attn_cfg = 'BRISC/brisc.yaml' if 'BRISC' in config_path else 'CAMUS/camus.yaml'
    bcfg = load_config(str(PKG_ROOT / 'configs' / attn_cfg))
    bcfg['label_frac'] = LABEL_FRAC   # -> _lf<pct> checkpoint suffix (Phase B)
    root = None
    for nm in root_names:
        c = [p for p in Path('/kaggle/input').rglob(nm) if p.is_dir()]
        if c:
            root = str(c[0]); break
    if root is None:
        raise FileNotFoundError(f'none of {root_names} under /kaggle/input')
    bcfg['data_root'] = root
    # exact phase-correct ATTENTION name first (e.g. camus_lf10_best.pt):
    ck = f"{bcfg['dataset'].lower()}{model_suffix(bcfg)}_best.pt"
    cands = list(Path('/kaggle/input').rglob(ck))
    if not cands:
        # lenient: any ATTENTION ckpt for this dataset (EXCLUDE the lite net):
        cands = sorted((p for p in Path('/kaggle/input').rglob(f"{bcfg['dataset'].lower()}*_best.pt")
                        if 'lite' not in p.name.lower()),
                       key=lambda p: p.stat().st_mtime, reverse=True)
        if cands:
            print(f"  [warn] exact ckpt {ck!r} not attached -- using {cands[0].name!r} "
                  f"instead (verify this is the Phase-B attention checkpoint you meant).")
    if not cands:
        near = [str(x) for x in Path('/kaggle/input').rglob(f"*{bcfg['dataset'].lower()}*.pt")]
        raise FileNotFoundError(
            f'attention checkpoint {ck!r} (or any non-lite {bcfg["dataset"].lower()}*_best.pt) '
            f'not found under /kaggle/input. Similar .pt files: {near or ["NONE"]}. '
            f'Attach the Phase-B ATTENTION checkpoint dataset.')
    cfg['baseline_checkpoint'] = str(cands[0])
    tr, va, te = precompute_init_masks(
        baseline_cfg=bcfg, baseline_checkpoint=cfg['baseline_checkpoint'],
        target_class=cfg['target_class'], min_area_fraction=cfg.get('min_area_fraction', 0.01),
        tumor_type_filter=cfg.get('tumor_type_filter'))
    SAMPLES[label] = dict(train=tr, val=va, test=te)
    CFGS[label] = cfg
    print(f'  {label}: train {len(tr)} | val {len(va)} | test {len(te)} | ckpt {Path(cfg["baseline_checkpoint"]).name}')
    return cfg


def _entropy(p):
    pc = np.clip(p.astype(np.float32), 1e-6, 1 - 1e-6)
    return float((-(pc*np.log2(pc) + (1-pc)*np.log2(1-pc))).mean())


def full_report(label, n_diag=60):
    """Full EDA + diagnostic for one class. Prints, plots, returns a metrics row."""
    import matplotlib.pyplot as plt
    s = SAMPLES[label]; cfg = CFGS[label]
    val = s['val']
    print('\n' + '=' * 78 + f'\n{label}\n' + '=' * 78)

    # ---- §3 Dataset EDA ----
    area = np.array([float(x['gt_mask'].mean()) for x in val])         # GT area fraction
    ncc  = np.array([ndi.label(x['gt_mask'], structure=STRUCT)[1] for x in val])
    imean = np.array([float(x['image'].mean()) for x in val])
    views, phases = {}, {}
    for x in val:
        views[x.get('view', '')]  = views.get(x.get('view', ''), 0) + 1
        phases[x.get('phase', '')] = phases.get(x.get('phase', ''), 0) + 1
    multi_cc = float((ncc > 1).mean())
    print(f'[dataset] n(train/val/test)={len(s["train"])}/{len(val)}/{len(s["test"])} | '
          f'GT area%: mean {area.mean()*100:.2f} med {np.median(area)*100:.2f} '
          f'[{area.min()*100:.2f},{area.max()*100:.2f}] | multi-blob GT: {multi_cc*100:.1f}% | '
          f'img mean {imean.mean():.3f}')
    if any(views):  print(f'[dataset] views={views} phases={phases}')

    # ---- §4 Lite U-Net baseline ----
    dice = np.array([dice_score(x['init_mask'], x['gt_mask']) for x in val])
    iou  = dice / (2 - dice + 1e-8)
    order = np.argsort(dice)
    worst = order[:5]
    print(f'[lite] init Dice: mean {dice.mean():.4f} med {np.median(dice):.4f} '
          f'p10 {np.percentile(dice,10):.4f} min {dice.min():.4f} | '
          f'IoU mean {iou.mean():.4f} | worst-5 Dice {np.round(dice[worst],3).tolist()}')
    pm = prob_map_informativeness(val, n_samples=n_diag, label=label)
    ent = float(np.mean([_entropy(np.asarray(x['prob_map'])) for x in val[:n_diag]]))
    prob_state = 'USABLE' if pm['uncertain_band_frac'] > 0.01 else 'INERT'
    print(f'[lite] prob_map: band_frac {pm["uncertain_band_frac"]:.4f} | mean entropy {ent:.4f} -> {prob_state}')

    # ---- §5 Error decomposition ----
    et = error_type_audit(val, n_samples=n_diag, label=label)

    # ---- §6 DRL compatibility ----
    hr = headroom_report(
        val, n_points=cfg['n_points'], cont_sectors=cfg.get('cont_sectors', 12),
        disp_px=cfg['disp_px'], spline_smooth=cfg['spline_smooth'],
        n_samples=n_diag, label=label, attention_dice=ATTN_DICE.get(label))

    # ---- plots ----
    fig, ax = plt.subplots(2, 3, figsize=(15, 8))
    ax[0,0].hist(area*100, bins=30, color='C0'); ax[0,0].set(title=f'{label}: GT area %', xlabel='% of image')
    ax[0,1].hist(dice, bins=30, color='C2'); ax[0,1].axvline(dice.mean(), color='k', ls='--')
    ax[0,1].set(title=f'init Dice (mean {dice.mean():.3f})', xlabel='Dice')
    allp = np.concatenate([np.asarray(x['prob_map']).ravel().astype(np.float32) for x in val[:20]])
    ax[0,2].hist(allp, bins=50, color='C3', log=True)
    ax[0,2].axvspan(0.35, 0.65, color='orange', alpha=0.25)
    ax[0,2].set(title=f'prob_map values ({prob_state})', xlabel='p (orange=gate band)')
    ax[1,0].bar([str(k) for k in sorted(set(ncc))], [int((ncc==k).sum()) for k in sorted(set(ncc))], color='C4')
    ax[1,0].set(title='GT connected components', xlabel='# blobs')
    ax[1,1].bar(['boundary','topology','interior'],
                [et['boundary_frac'], et['topology_frac'], et['interior_frac']],
                color=['C2','C3','C1'])
    ax[1,1].set(title='error decomposition', ylim=(0,1))
    bars = ['baseline','repr cap','oracle(GT)']
    vals = [hr['baseline_init_dice'], hr['contour_repr_ceiling'], hr['oracle_greedy_ceiling']]
    if ATTN_DICE.get(label) is not None:
        bars.append('attn'); vals.append(ATTN_DICE[label])
    ax[1,2].bar(bars, vals, color='C0'); ax[1,2].set(title='ceilings vs baseline', ylim=(min(vals)-0.05,1.0))
    ax[1,2].axhline(hr['baseline_init_dice'], color='k', ls='--', lw=0.8)
    plt.suptitle(f'{label} — EDA + diagnostic', fontweight='bold'); plt.tight_layout()
    plt.savefig(f'/kaggle/working/eda_{label}.png', dpi=120); plt.show()

    # qualitative: the 3 worst lite cases (where headroom is)
    fig2, ax2 = plt.subplots(3, 4, figsize=(13, 9))
    for r, idx in enumerate(worst[:3]):
        x = val[int(idx)]
        ax2[r,0].imshow(x['image'], cmap='gray'); ax2[r,0].set_ylabel(f'Dice {dice[idx]:.2f}')
        ax2[r,1].imshow(x['gt_mask'], cmap='Greens'); ax2[r,1].set_title('GT' if r==0 else '')
        ax2[r,2].imshow(x['init_mask'], cmap='Blues'); ax2[r,2].set_title('lite init' if r==0 else '')
        ax2[r,3].imshow(np.asarray(x['prob_map']).astype(np.float32), cmap='magma', vmin=0, vmax=1)
        ax2[r,3].set_title('prob_map' if r==0 else '')
        for c in range(4): ax2[r,c].set_xticks([]); ax2[r,c].set_yticks([])
    plt.suptitle(f'{label} — 3 worst lite cases (largest headroom)', fontweight='bold'); plt.tight_layout()
    plt.savefig(f'/kaggle/working/eda_{label}_worst.png', dpi=120); plt.show()

    realistic = hr.get('realistic_headroom_estimate')
    row = dict(
        label=label, n_val=len(val),
        gt_area_pct=round(area.mean()*100, 2), multi_blob_pct=round(multi_cc*100, 1),
        baseline_dice=round(hr['baseline_init_dice'], 4),
        baseline_dice_p10=round(float(np.percentile(dice,10)), 4),
        prob_state=prob_state, band_frac=round(pm['uncertain_band_frac'], 4),
        boundary_frac=round(et['boundary_frac'], 2),
        topology_interior_frac=round(et['topology_frac']+et['interior_frac'], 2),
        contour_repr_ceiling=round(hr['contour_repr_ceiling'], 4),
        oracle_ceiling_GTpriv=round(hr['oracle_greedy_ceiling'], 4),
        attn_dice=ATTN_DICE.get(label),
        realistic_headroom=(round(realistic, 4) if realistic is not None else None),
        pillar4_go=(pm['uncertain_band_frac'] > 0.01 and et['boundary_frac'] > 0.5),
    )
    RESULTS[label] = row
    return row

## 2 · Warm-start all CAMUS classes (Phase-B ATTENTION net)
~5 min/class. Uses the label_frac=0.10 attention checkpoint (`camus_lf10_best.pt`).


In [ ]:
print('Warm-starting CAMUS classes...')
load_class('CAMUS/DRL/camus_drl_c1.yaml', ['CAMUS'], 'CAMUS_LV_endo')
load_class('CAMUS/DRL/camus_drl_c2.yaml', ['CAMUS'], 'CAMUS_LV_epi')
load_class('CAMUS/DRL/camus_drl_c3.yaml', ['CAMUS'], 'CAMUS_LA')

## 3-6 · Per-class EDA + diagnostic report (CAMUS)

In [ ]:
for lab in ['CAMUS_LV_endo', 'CAMUS_LV_epi', 'CAMUS_LA']:
    full_report(lab)

## (optional) BRISC

Off by default — BRISC currently has no realistic headroom (lite baseline already above the attention
competitor). Set `RUN_BRISC = True` to include the generic-tumour class (~10 min warm-start).

In [ ]:
RUN_BRISC = False
if RUN_BRISC:
    load_class('BRISC/DRL/brisc_drl_tumor.yaml', ['brisc2025', 'BRISC'], 'BRISC_tumor')
    full_report('BRISC_tumor')
else:
    print('Skipped (RUN_BRISC = False).')

## 7 · Master summary + per-class recommendation

The actionable bottom line. `realistic_headroom` (attention Dice − baseline Dice) is the honest gain
ceiling; `prob_state` decides whether the uncertainty gate can function; `boundary_frac` decides whether
contour deformation is even the right tool. The recommendation column combines all three.

In [ ]:
def recommend(r):
    rh = r['realistic_headroom']
    if rh is None:
        return 'no attn_dice set — ceiling-only'
    if rh <= 0.005:
        return 'SKIP — no realistic headroom (baseline >= competitor)'
    if r['boundary_frac'] <= 0.5:
        return f'RISKY — only {r["boundary_frac"]:.0%} boundary error; contour can\'t fix the rest'
    gate = ('gate ON (prob_map usable)' if r['prob_state'] == 'USABLE'
            else 'gate OFF or retrain lite w/ label smoothing (prob_map INERT)')
    strength = 'STRONG' if rh > 0.05 else 'GOOD' if rh > 0.02 else 'MARGINAL'
    return f'TRAIN [{strength}, +{rh:.3f} headroom] — {gate}'

rows = list(RESULTS.values())
summary = pd.DataFrame(rows).set_index('label')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
display(summary)
summary.to_csv('/kaggle/working/eda_diagnostic_summary.csv')
print('\nSaved /kaggle/working/eda_diagnostic_summary.csv')

def oracle_recommend(r):
    oh = r['oracle_ceiling_GTpriv'] - r['baseline_dice']   # GT-oracle headroom (the real ceiling)
    if oh <= 0.005:
        return f'NO CEILING — GT-oracle headroom +{oh:.3f}; contour refinement cannot beat this baseline'
    if r['boundary_frac'] <= 0.5:
        return f'RISKY — only {r["boundary_frac"]:.0%} boundary error; contour can\'t fix the rest'
    strength = 'STRONG' if oh > 0.05 else 'GOOD' if oh > 0.02 else 'MARGINAL'
    gate = ('gate ON (prob_map usable)' if r['prob_state'] == 'USABLE'
            else 'gate OFF or retrain w/ label smoothing (prob_map INERT)')
    return f'TRAIN [{strength}, +{oh:.3f} GT-oracle headroom] — {gate}'

print('\n' + '='*78 + '\nPER-CLASS RECOMMENDATION (ranked by GT-oracle headroom)\n' + '='*78)
ranked = sorted(rows, key=lambda r: r['oracle_ceiling_GTpriv'] - r['baseline_dice'], reverse=True)
for r in ranked:
    oh = r['oracle_ceiling_GTpriv'] - r['baseline_dice']
    print(f"{r['label']:16s} | baseline {r['baseline_dice']:.4f} -> oracle-ceiling "
          f"{r['oracle_ceiling_GTpriv']:.4f} (+{oh:.4f}) | {r['prob_state']:6s} "
          f"| boundary {r['boundary_frac']:.2f} | {oracle_recommend(r)}")
if ranked:
    top = ranked[0]
    top_oh = top['oracle_ceiling_GTpriv'] - top['baseline_dice']
    print(f"\n>> Best single bet this week: {top['label']} (+{top_oh:.3f} GT-oracle headroom).")